

# 🎬 Lights, Camera, Agents! — the lab
### From Zero to Hero: Building & Evaluating Multi-Agent Systems

**Tonight's production: _Blog-to-Thread Studio_.**

You wrote a blog post. It's good. Eleven people read it. 😅
The internet lives on X (Twitter), so you'll hire a **crew of AI agents** to turn any blog URL into a
punchy, fact-checked X thread and drop it into your **Typefully** queue. Your **eval suite** will tell you
whether the crew did a good job.

By the end of this notebook you will have:

- 🤖 built **agents** with Microsoft Agent Framework: a model, a brief (instructions) and props (tools)
- 🔧 given them **real tools**: a web scraper and a social-media scheduler
- 🔁 wired up a **reflection loop** (writer ⇄ critic) and a **multi-agent workflow**
- 🖥️ watched it all run live in **DevUI**
- ⚖️ scored the output with **code checks + an LLM judge**

## 🗺️ The crew you'll build



### How this notebook works

| Icon | Meaning |
|---|---|
| ▶️ | Just run the cell (`Shift + Enter`). |
| ✍️ **YOUR TURN** | Fill in the `TODO`s. A collapsed **🎞️ solution** sits right below; try for 5 minutes before peeking. |
| 🎤 **Pause & ponder** | A question to discuss with your neighbour. There's no code. |
| 🎟️ **Checkpoint** | A cell that tells you whether you're on track. |

### 🗓️ Shooting schedule (≈ 60 min)

| Scene | What you'll do | ⏱️ |
|---|---|---|
| **SC 1** · Keys & clapper | Load secrets, make your first model call | 5 |
| **SC 2** · The Reader | Your first agent with a real tool (demo, just run it) | 8 |
| **SC 3** · The Writer | ✍️ Write a brief that outputs JSON | 8 |
| **SC 4** · The Critic | ✍️ Build a code-check tool and a reflection loop | 12 |
| **SC 5** · The Publisher | ✍️ Build a Typefully tool and rehearse in dry-run | 5 |
| **SC 6** · Premiere | ✍️ Wire the whole crew into one workflow and ship 🎉 | 7 |
| **SC 7** · Dailies | Watch the whole crew in DevUI | 5 |
| **SC 8** · The reviews are in | ✍️ Score it with code checks and an LLM judge | 10 |

> 💡 **In Colab:** `File → Save a copy in Drive` first, so your edits are saved.

---
# 🎬 SC 1 · Keys & clapper

### 🎟️ Three tickets (only the first one is required)

| Ticket | Why | Where | Required? |
|---|---|---|---|
| `GEMINI_API_KEY` | The **talent**: free Gemini models | [aistudio.google.com/apikey](https://aistudio.google.com/apikey) → **Create API key** (free tier, no card). In Colab you can also use 🔑 → **Gemini API keys → Import key** (saved as `GOOGLE_API_KEY`, which works too) | ✅ Yes |
| `FIRECRAWL_API_KEY` | The **scout**: scrapes clean markdown from any web page | [firecrawl.dev](https://www.firecrawl.dev/) → sign up → API keys (free credits) | Optional: falls back to the free Jina reader |
| `TYPEFULLY_API_KEY` | The **distributor**: drafts and schedules posts on X | [typefully.com](https://typefully.com) → Settings → API (connect an X account) | Optional: SC 5–6 run in *dry-run* mode without it |

### 🔑 Add them to Colab Secrets
1. Click the **🔑 key icon** in the left sidebar.
2. **+ Add new secret** → name it exactly as above → paste the value.
3. Toggle **Notebook access** on.

> ⚠️ **Never paste keys into a code cell.** Notebooks get shared, and so do your keys.
>
> 🏠 **Running locally instead?** Put the same names in a `.env` file next to the notebook.
> To use a local model, set `LAB_PROVIDER=ollama` (it needs [Ollama](https://ollama.com) and `ollama pull qwen2.5:7b`).

### ▶️ Install the gear
Microsoft Agent Framework (`agent-framework-*`), its DevUI and the Firecrawl SDK. Versions are pinned, so the lab
behaves the same for everyone. It takes about a minute.

In [ ]:
import importlib.metadata as meta, subprocess, sys

PACKAGES = [
    "agent-framework-core==1.21.0",      # Agent, tools, workflows, evals
    "agent-framework-openai==1.16.0",    # OpenAI-compatible model clients (Gemini, Ollama, ...)
    "agent-framework-devui==1.0.0b261008",  # the "video village" UI
    "firecrawl-py==4.50.0",              # web scraping
    "httpx<1",
]

def _pinned_ok() -> bool:
    for spec in PACKAGES:
        if "==" not in spec:
            continue
        name, version = spec.split("==")
        try:
            if meta.version(name) != version:
                return False
        except meta.PackageNotFoundError:
            return False
    return True

if not _pinned_ok():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PACKAGES], check=True)
print("✅ Gear installed")

### ▶️ Load the keys and pick the talent

**Google's Gemini API** has a free (rate-limited) tier and an OpenAI-compatible endpoint.
So we use MAF's `OpenAIChatCompletionClient` and just point its `base_url` at Gemini.
(The cell also adds a little backstage plumbing that Gemini needs for tool calls; you can ignore it.)
That same client works with Ollama, Groq, OpenRouter or any OpenAI-compatible server.

In [ ]:
import os, re, json, asyncio, warnings, textwrap
from typing import Annotated
from getpass import getpass

from pydantic import BaseModel, Field
from IPython.display import display, HTML, Markdown

warnings.filterwarnings("ignore")  # MAF marks evals as experimental; keep the output tidy
IN_COLAB = "google.colab" in sys.modules

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

def get_secret(name: str, required: bool = False) -> str | None:
    """Colab Secrets → environment / .env → (if required) a hidden prompt."""
    value = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            value = userdata.get(name)
        except Exception:
            value = None
    value = value or os.getenv(name)
    if not value and required:
        value = getpass(f"🔑 Paste your {name}: ")
    return (value or "").strip() or None

PROVIDERS = {
    #            OpenAI-compatible endpoint                                  default model        key names
    "gemini": ("https://generativelanguage.googleapis.com/v1beta/openai/", "gemini-3.5-flash-lite", ("GEMINI_API_KEY", "GOOGLE_API_KEY")),
    "ollama": ("http://localhost:11434/v1",                                "qwen2.5:7b",        ()),
}
PROVIDER = os.getenv("LAB_PROVIDER", "gemini")
BASE_URL, MODEL, KEY_NAMES = PROVIDERS[PROVIDER]
MODEL = os.getenv("LAB_MODEL", MODEL)
API_KEY = "not-needed"
if KEY_NAMES:
    API_KEY = next((k for k in map(get_secret, KEY_NAMES) if k), None) or get_secret(KEY_NAMES[0], required=True)

FIRECRAWL_API_KEY = get_secret("FIRECRAWL_API_KEY")
TYPEFULLY_API_KEY = get_secret("TYPEFULLY_API_KEY")

# ── Backstage plumbing: Gemini 3 "thought signatures" ─────────────────────────────
# Gemini attaches an opaque `thought_signature` to every tool call and expects it back on the next
# request. Generic OpenAI clients drop it, so this tiny transport remembers each signature and
# re-attaches it (and cues the next agent's turn in a crew). You never need to touch it.
import httpx
from openai import AsyncOpenAI

class ThoughtSignatureTransport(httpx.AsyncBaseTransport):
    FALLBACK = "skip_thought_signature_validator"   # Google's documented placeholder

    def __init__(self):
        self._inner = httpx.AsyncHTTPTransport(retries=2)
        self._signatures: dict[str, str] = {}

    async def handle_async_request(self, request: httpx.Request) -> httpx.Response:
        if request.method == "POST" and request.url.path.endswith("/chat/completions"):
            body = json.loads(request.content or b"{}")
            patched = False
            for message in body.get("messages", []):
                for call in message.get("tool_calls") or []:
                    if "extra_content" not in call:
                        sig = self._signatures.get(call.get("id"), self.FALLBACK)
                        call["extra_content"] = {"google": {"thought_signature": sig}}
                        patched = True
            # Gemini refuses a conversation that ends on the model's own turn, which is exactly what a
            # sequential crew hands the next agent. A one-line cue makes it the next agent's turn.
            msgs = body.get("messages") or []
            if msgs and msgs[-1].get("role") == "assistant":
                msgs.append({"role": "user", "content": "Your turn: do your job on the conversation above."})
                patched = True
            if patched:
                headers = {k: v for k, v in request.headers.items() if k.lower() != "content-length"}
                request = httpx.Request(request.method, request.url, headers=headers, json=body)
        response = await self._inner.handle_async_request(request)
        if "application/json" not in response.headers.get("content-type", ""):
            return response
        data = await response.aread()
        try:   # error bodies (429/503) can be JSON lists, so never let bookkeeping crash a take
            payload = json.loads(data)
            choices = payload.get("choices", []) if isinstance(payload, dict) else []
            for choice in choices:
                for call in (choice.get("message") or {}).get("tool_calls") or []:
                    sig = (call.get("extra_content") or {}).get("google", {}).get("thought_signature")
                    if sig and call.get("id"):
                        self._signatures[call["id"]] = sig
        except Exception:
            pass
        headers = [(k, v) for k, v in response.headers.items() if k.lower() not in ("content-encoding", "content-length")]
        return httpx.Response(response.status_code, headers=headers, content=data, request=request)

OPENAI_CLIENT = AsyncOpenAI(
    api_key=API_KEY, base_url=BASE_URL, max_retries=5,   # retries back off politely on free-tier 429s
    http_client=httpx.AsyncClient(transport=ThoughtSignatureTransport(), timeout=120),
)

print(f"🎭 Talent  : {MODEL}  via {PROVIDER} ({BASE_URL})")
print(f"🔭 Scout   : {'Firecrawl' if FIRECRAWL_API_KEY else 'Jina reader (no key needed)'}")
print(f"📣 Distrib.: {'Typefully' if TYPEFULLY_API_KEY else 'dry-run (no TYPEFULLY_API_KEY)'}")

### ▶️ Action! Your first agent

An **`Agent`** = a **client** (the talent) + **instructions** (the brief) + optional **tools** (the props).
`await agent.run(...)` performs one take and returns an `AgentResponse`. Read the reply from `.text`.

In [ ]:
from agent_framework import Agent, tool, WorkflowBuilder
from agent_framework.openai import OpenAIChatCompletionClient

# OPENAI_CLIENT = an OpenAI SDK client pointed at BASE_URL with your key (built in the cell above)
client = OpenAIChatCompletionClient(model=MODEL, async_client=OPENAI_CLIENT)

director = Agent(
    client,
    name="Director",
    instructions="You are an over-excited film director. Answer in ONE short line with one emoji.",
)

response = await director.run("We're about to build AI agents. Say the line that starts every take!")
print(response.text)

🎟️ **Checkpoint:** you should see a one-liner like *"Lights, camera, ACTION! 🎬"*.

- **`400` / `API key not valid`?** Re-copy the key from AI Studio and check the secret's name and *Notebook access*.
- **`429`?** You hit the free-tier rate limit; wait a minute and re-run the cell.

> 🎤 **Pause & ponder:** Is `director` an *agent* yet? It has a brief, but no tools and no goal to loop
> towards. Hold that thought until SC 2.

---
# 📖 SC 2 · The Reader: your first real agent
*Demo scene: just run the cells and read along.*

A model on its own can't open a URL. We hand it a **prop**: a plain Python function.
MAF reads the function's **name, docstring and type hints** and turns them into a JSON schema the model can see.
When the model decides to use the tool, MAF runs your Python and feeds the result back. That's the **agent loop**:



In [ ]:
import httpx

MAX_PAGE_CHARS = 10_000   # free-tier models have ~8k-token input limits, so we trim long pages

SAMPLE_URL = "https://myblog.dev/first-internship"
SAMPLE_BLOG = """
# What my first internship actually taught me
I spent 12 weeks as a software intern at a 40-person fintech startup in Bengaluru.
I shipped 3 features to production; the biggest one cut invoice-processing time from 2 days to 4 hours.
The best advice came from my manager in week 1: "Ask after 30 minutes stuck, not after 3 hours."
I broke the staging database once. Nobody yelled; we wrote a 1-page post-mortem and a backup script.
Code review felt brutal at first: my first pull request got 47 comments.
By week 10 I was reviewing other interns' code.
Biggest lesson: writing things down (design notes, PR descriptions, weekly updates) mattered more than clever code.
I got a return offer, and I'm going back after graduation.
"""

def _scrape(url: str) -> str:
    if url.rstrip("/") == SAMPLE_URL:
        return SAMPLE_BLOG
    if FIRECRAWL_API_KEY:
        from firecrawl import Firecrawl
        doc = Firecrawl(api_key=FIRECRAWL_API_KEY).scrape(url, formats=["markdown"], only_main_content=True)
        return doc.markdown or ""
    # Free fallback: Jina's reader turns any page into markdown, no key needed
    return httpx.get(f"https://r.jina.ai/{url}", timeout=60, follow_redirects=True).text

@tool
def scrape_url(url: Annotated[str, Field(description="Full http(s) URL of the blog post to read")]) -> str:
    """Fetch a web page and return its main content as markdown."""
    try:
        text = _scrape(url)
    except Exception as e:
        return f"ERROR: could not fetch {url}: {e}"
    if len(text) > MAX_PAGE_CHARS:
        text = text[:MAX_PAGE_CHARS] + "\n\n[...truncated...]"
    return text

# This is exactly what the model sees about your tool:
print(json.dumps(scrape_url.to_json_schema_spec(), indent=2))

👀 Notice that the **docstring became the tool's description** and the type hint became a JSON schema.
The model never sees your code, only this "label on the prop". Good tool names and descriptions are prompt engineering too.

Now cast the Reader. Its job is to turn a messy web page into a tight **brief** for the Writer.

In [ ]:
READER_INSTRUCTIONS = """
You are the Reader in a film crew that turns blog posts into X threads.
1. Fetch the web page at the URL you are given and read its full text.
2. Write a READER'S BRIEF in exactly this shape:
   TITLE: <the post's title>
   URL: <the url>
   THESIS: <one sentence: the single big idea>
   KEY POINTS: 5-7 bullets, each with a concrete fact, number or example from the post
   QUOTABLE LINES: 1-3 short lines copied verbatim from the post
Use ONLY information from the page. If the page cannot be fetched, say so and stop.
"""

reader = Agent(client, name="Reader", instructions=READER_INSTRUCTIONS, tools=[scrape_url])

def show_tool_calls(response) -> None:
    """Print the props the agent picked up during a take."""
    for msg in response.messages:
        for c in msg.contents:
            if c.type == "function_call":
                print(f"🔧 {c.name}({c.arguments})")
            elif c.type == "function_result":
                print(f"   ↳ returned {len(str(c.result)):,} chars")

### 🎯 Pick your blog
Use the sample (`https://myblog.dev/first-internship`, built in and always works) or **paste any real blog URL**:
your own, a friend's, or an engineering blog you like. Short-to-medium posts work best.

In [ ]:
BLOG_URL = "https://myblog.dev/first-internship"   # ← swap in a real blog URL!

brief_response = await reader.run(f"Read this blog post and write the brief: {BLOG_URL}")
show_tool_calls(brief_response)
brief = brief_response.text
display(Markdown(brief))

# Every brief you make joins the eval set in SC 8, next to the fixed test briefs
MY_BRIEFS = globals().get("MY_BRIEFS", [])
if "TITLE:" in brief and brief not in MY_BRIEFS:
    MY_BRIEFS.append(brief)
print(f"📼 {len(MY_BRIEFS)} brief(s) of yours saved for the evals")

🎟️ **Checkpoint:** you should see a `🔧 scrape_url(...)` line followed by a brief with TITLE, THESIS and KEY POINTS.
**That's an agent:** the model *decided* to call a tool, observed the result, then produced its answer.

> 🎤 **Pause & ponder:** We trimmed the page to 10,000 characters. What could go wrong with a 50,000-character blog
> post? What would you do in production? (Think chunking, summarising in passes, or a model with a bigger context window.)

---
# ✍️ SC 3 · The Writer: a brief that outputs JSON

Downstream *code* needs to count characters, check hashtags and send posts to an API. So free text won't do;
we want **structured output**. Describe the shape with a Pydantic model, pass it as `response_format`,
and the reply arrives as a typed Python object in `response.value`.

In [ ]:
class Thread(BaseModel):
    """An X (Twitter) thread."""
    posts: list[str] = Field(description="The posts of the thread, in order. posts[0] is the hook.")

def storyboard(posts: list[str], title: str = "Storyboard") -> None:
    """Render a thread as a film strip, with character counts (green ≤ 280, red > 280)."""
    frames = "".join(
        f"<div style='flex:0 0 220px;background:#FBF5EA;color:#261B2B;border:2px solid #261B2B;"
        f"border-radius:10px;padding:10px;font:14px/1.4 sans-serif;box-shadow:4px 4px 0 #261B2B'>"
        f"<div style='font:bold 12px monospace;color:#5B4A60'>POST {i + 1} · "
        f"<span style='color:{'#2e7d32' if len(p) <= 280 else '#c62828'}'>{len(p)}/280</span></div>"
        f"<div style='margin-top:6px;white-space:pre-wrap'>{p}</div></div>"
        for i, p in enumerate(posts)
    )
    display(HTML(
        f"<div style='background:#2E1838;padding:14px;border-radius:14px'>"
        f"<div style='color:#F0A830;font:bold 14px monospace;margin-bottom:10px'>🎞️ {title}</div>"
        f"<div style='display:flex;gap:14px;overflow-x:auto;padding-bottom:6px'>{frames}</div></div>"
    ))

### ✍️ YOUR TURN: write the Writer's brief

A vague brief gets you a vague thread. Turn the **house rules** below into instructions:

| # | House rule |
|---|---|
| 1 | **5 to 7** posts |
| 2 | Post 1 is a **hook**: a memorable line that stops a scrolling reader. It can be provocative, a question, an exclamation or a quotable line. No "🧵" or "Thread:". |
| 3 | Each post **≤ 280 characters** |
| 4 | **Only facts from the brief**: no invented numbers, names or quotes |
| 5 | **Max 2 hashtags** in the whole thread, only in the last post |
| 6 | The last post links back to the blog URL and invites replies |
| 7 | If **director's notes** are present, fix every note and change nothing else |

In [ ]:
WRITER_INSTRUCTIONS = """
You are the Writer in a film crew that turns blog posts into X (Twitter) threads.
You receive a reader's brief, and sometimes DIRECTOR'S NOTES from a previous take.

TODO: turn the 7 house rules above into clear, numbered instructions.
"""

writer = Agent(
    client,
    name="Writer",
    instructions=WRITER_INSTRUCTIONS,
    default_options={"response_format": ...},   # ✍️ TODO: which Pydantic model describes the output?
)

<details>
<summary><b>🎞️ Stuck? Peek at the solution</b> (try for 5 minutes first)</summary>

<p><i>Hint: numbered rules beat paragraphs. Be explicit about numbers (5-7, 280, 2), and say what NOT to do.</i></p>

```python
WRITER_INSTRUCTIONS = """
You are the Writer in a film crew that turns blog posts into X (Twitter) threads.
You receive a reader's brief, and sometimes DIRECTOR'S NOTES from a previous take.

House rules - follow every one:
1. Write 5 to 7 posts in total.
2. Post 1 is the hook: a memorable line that makes a scrolling reader stop and pay attention.
   It can be provocative, a question, an exclamation or a quotable line, built from the brief.
   Never start with "Thread:" or the 🧵 emoji.
3. Every post must be 280 characters or fewer. Aim for 180-250.
4. Use ONLY facts that appear in the brief. Never invent numbers, names, quotes or outcomes.
5. One idea per post. Plain, conversational language; at most one emoji per post.
6. Use at most 2 hashtags in the whole thread, and only in the last post.
7. The last post links to the original blog URL and invites readers to reply.
If DIRECTOR'S NOTES are present, fix every note and change nothing else.
"""

writer = Agent(client, name="Writer", instructions=WRITER_INSTRUCTIONS,
               default_options={"response_format": Thread})
```

</details>

In [ ]:
thread_response = await writer.run(f"Write the thread for this brief:\n\n{brief}")
thread: Thread = thread_response.value
storyboard(thread.posts, "Writer · take 1")

🎟️ **Checkpoint:** a film strip of 5–7 posts. Are all the counters green? Is post 1 actually a hook?
If a post is red, don't worry: that's exactly what the Critic is for. 😉

---
# 🧐 SC 4 · The Critic: a reflection loop

LLMs are bad at counting characters but good at judging tone. Python is the opposite.
So the Critic gets **both**: a deterministic **`check_thread` tool** for the countable rules, and its own judgement
for everything else. Then we loop **writer → critic → writer**, capped at 3 takes.



### ✍️ YOUR TURN (1/2): the code-check prop
Fill in the three checks. Each problem becomes a short, **actionable** message, because the Writer will read these notes.

In [ ]:
MIN_POSTS, MAX_POSTS = 5, 7
MAX_CHARS_PER_POST = 280
MAX_HASHTAGS = 2

def check_thread_rules(posts: list[str]) -> dict:
    """Check a thread against the countable house rules."""
    issues = []
    # ✍️ TODO 1: too few or too many posts?        -> issues.append("Thread has N posts; it needs 5-7.")
    # ✍️ TODO 2: any post over MAX_CHARS_PER_POST? -> say WHICH post and by HOW MUCH
    # ✍️ TODO 3: more than MAX_HASHTAGS hashtags in the whole thread?  (hint: re.findall(r"#\w+", text))
    return {"ok": not issues, "issues": issues}

# Wrap the plain function as a tool the Critic can call
check_thread = tool(check_thread_rules, name="check_thread",
                    description="Check X-thread posts against the countable house rules "
                                "(post count, 280-char limit, hashtag budget). Returns ok + issues.")

<details>
<summary><b>🎞️ Stuck? Peek at the solution</b> (try for 5 minutes first)</summary>

```python
MIN_POSTS, MAX_POSTS = 5, 7
MAX_CHARS_PER_POST = 280
MAX_HASHTAGS = 2

def check_thread_rules(posts: list[str]) -> dict:
    """Check a thread against the countable house rules."""
    issues = []
    if not MIN_POSTS <= len(posts) <= MAX_POSTS:
        issues.append(f"Thread has {len(posts)} posts; it needs {MIN_POSTS}-{MAX_POSTS}.")
    for i, post in enumerate(posts, start=1):
        if len(post) > MAX_CHARS_PER_POST:
            issues.append(f"Post {i} is {len(post)} chars; cut {len(post) - MAX_CHARS_PER_POST}+ chars.")
    hashtags = re.findall(r"#\w+", " ".join(posts))
    if len(hashtags) > MAX_HASHTAGS:
        issues.append(f"{len(hashtags)} hashtags used ({' '.join(hashtags)}); keep at most {MAX_HASHTAGS}.")
    return {"ok": not issues, "issues": issues}

check_thread = tool(check_thread_rules, name="check_thread",
                    description="Check X-thread posts against the countable house rules "
                                "(post count, 280-char limit, hashtag budget). Returns ok + issues.")
```

</details>

In [ ]:
# 🎟️ Checkpoint: these unit tests should all pass
good = ["Hook!"] + ["A solid post."] * 4
assert check_thread_rules(good)["ok"], "a clean 5-post thread should pass"
assert not check_thread_rules(["Too short"])["ok"], "1 post should fail the count rule"
assert not check_thread_rules(good + ["x" * 300])["ok"], "a 300-char post should fail"
assert not check_thread_rules(good[:4] + ["#a #b #c"])["ok"], "3 hashtags should fail"
print("✅ check_thread works. Here's what it says about the Writer's take 1:")
print(check_thread_rules(thread.posts))

### ✍️ YOUR TURN (2/2): cast the Critic
The Critic returns a `Review`. Write its instructions so that it:
1. **always** runs the house-rules checker first and treats every issue it reports as a must-fix,
2. then judges what code can't: does post 1 really hook (memorable, makes a scroller stop)? Is every claim backed by the brief?
3. approves only if the checker passes **and** it has no must-fix notes,
4. writes **short, actionable** notes when it wants changes ("Post 3: cut the second sentence"),
   and a short **why it works** rationale when it approves, and
5. returns `final_posts`: the thread with any *small* fixes applied, ready to ship.

In [ ]:
class Review(BaseModel):
    approved: bool = Field(description="True only if the thread is ready to publish")
    notes: list[str] = Field(description="If not approved: short, actionable fixes for the Writer. "
                                         "If approved: 1-2 sentences on why the thread is ready.")
    final_posts: list[str] = Field(description="The thread to ship: the writer's posts with small safe fixes applied")

CRITIC_INSTRUCTIONS = """
You are the Critic: a sharp but fair script editor for X threads.
You receive the reader's brief and the writer's thread.

TODO: turn the 5 points above into numbered instructions.
"""

critic = Agent(
    client,
    name="Critic",
    instructions=CRITIC_INSTRUCTIONS,
    tools=[...],                                  # ✍️ TODO: which prop does the Critic need?
    default_options={"response_format": ...},     # ✍️ TODO: which model describes its output?
)

<details>
<summary><b>🎞️ Stuck? Peek at the solution</b> (try for 5 minutes first)</summary>

```python
class Review(BaseModel):
    approved: bool = Field(description="True only if the thread is ready to publish")
    notes: list[str] = Field(description="If not approved: short, actionable fixes for the Writer. "
                                         "If approved: 1-2 sentences on why the thread is ready.")
    final_posts: list[str] = Field(description="The thread to ship: the writer's posts with small safe fixes applied")

CRITIC_INSTRUCTIONS = """
You are the Critic: a sharp but fair script editor for X threads.
You receive the reader's brief and the writer's thread.
1. ALWAYS run the house-rules checker on the posts first. Every issue it reports is a must-fix.
2. Then judge what code cannot: Is post 1 a genuine hook - memorable, provocative, a question,
   an exclamation or a quotable line that would stop a scrolling reader? Is every claim supported
   by the brief? Is it one idea per post? Does the last post link to the blog?
3. approved = true only if the house-rules check passed AND you have no must-fix notes.
4. notes: never empty.
   - Not approved: short, actionable fixes, e.g. "Post 3: cut the second sentence".
   - Approved: 1-2 sentences on why the thread is ready (what makes the hook and the flow work).
5. final_posts: the thread to ship. Copy the writer's posts and apply only small safe fixes
   (trim words, remove extra hashtags). Never add facts that are not in the brief.
"""

critic = Agent(client, name="Critic", instructions=CRITIC_INSTRUCTIONS,
               tools=[check_thread], default_options={"response_format": Review})
```

</details>

### ▶️ Roll the reflection loop
This is the **reflection pattern** in ~15 lines of plain Python. The loop has a hard budget (`max_takes`), and
**code has the final say**: even if the Critic approves, a failing `check_thread` means another take.

In [ ]:
async def shoot(brief: str, max_takes: int = 3) -> Thread:
    notes = ""
    for take in range(1, max_takes + 1):
        prompt = f"BRIEF:\n{brief}"
        if notes:
            prompt += f"\n\nDIRECTOR'S NOTES from take {take - 1}:\n{notes}"
        draft: Thread = (await writer.run(prompt)).value

        review: Review = (await critic.run(
            f"BRIEF:\n{brief}\n\nWRITER'S THREAD:\n{draft.model_dump_json(indent=2)}"
        )).value
        hard = check_thread_rules(draft.posts)          # code has the final say
        approved = review.approved and hard["ok"]

        print(f"🎬 Take {take}: {'✅ approved' if approved else '❌ notes'}")
        for n in hard["issues"] + review.notes:
            print(f"   {'💬' if approved else '•'} {n}")
        if approved:
            return draft
        fixes = hard["issues"] + ([] if review.approved else review.notes)
        notes = "\n".join(fixes)
    print("⏱️ Out of takes. Shipping the best effort; a human should look at this one.")
    return draft

final_thread = await shoot(brief)
storyboard(final_thread.posts, "Final cut")

> 🎤 **Pause & ponder:**
> - Why cap it at 3 takes? (💸 cost, ⏱️ latency, and critics that never say "yes")
> - The Critic and the Writer use the **same model**. Is that a fair review? What would you change?

---
# 📣 SC 5 · The Publisher: a prop with real-world effects

Every crew needs someone to get the film into theatres. The **Publisher** is an agent with one prop:
a tool that drops the thread into your **Typefully** queue.

[Typefully](https://typefully.com) schedules posts to X, LinkedIn, Threads and Bluesky. Its
[v2 API](https://typefully.com/docs/api) takes a draft as JSON:

```json
POST https://api.typefully.com/v2/social-sets/{social_set_id}/drafts
Authorization: Bearer <TYPEFULLY_API_KEY>

{
  "draft_title": "My first internship",
  "share": true,
  "platforms": { "x": { "enabled": true, "posts": [ {"text": "post 1"}, {"text": "post 2"} ] } },
  "publish_at": "next-free-slot"        ← optional: omit it to save as a draft
}
```

🛡️ **Safety first:** by default we only **save a draft** (`SCHEDULE = False`), so nothing posts without you
reviewing it. No Typefully key? The tool runs in **dry-run** mode and shows you what it *would* send.

### ✍️ YOUR TURN: build the `payload`

In [ ]:
SCHEDULE = False   # True = queue in your next free Typefully slot; False = save as a draft only
TYPEFULLY_API = "https://api.typefully.com/v2"

def _typefully(method: str, path: str, **kwargs) -> dict:
    r = httpx.request(method, f"{TYPEFULLY_API}{path}", timeout=30,
                      headers={"Authorization": f"Bearer {TYPEFULLY_API_KEY}"}, **kwargs)
    r.raise_for_status()
    return r.json()

@tool
def create_typefully_draft(
    posts: Annotated[list[str], Field(description="The thread's posts, in order")],
    title: Annotated[str, Field(description="A short internal title for the draft")],
) -> dict:
    """Save an X thread as a Typefully draft (optionally scheduled). Returns the draft's status and URL."""
    payload = {
        # ✍️ TODO: "draft_title", "share", and "platforms" -> "x" -> enabled + posts [{"text": ...}, ...]
    }
    # ✍️ TODO: if SCHEDULE, add "publish_at": "next-free-slot"

    if not TYPEFULLY_API_KEY:
        return {"status": "dry-run", "would_send": payload}
    social_set_id = get_secret("TYPEFULLY_SOCIAL_SET_ID") or _typefully("GET", "/social-sets")["results"][0]["id"]
    draft = _typefully("POST", f"/social-sets/{social_set_id}/drafts", json=payload)
    return {"status": draft["status"], "url": draft.get("share_url") or draft.get("private_url")}

<details>
<summary><b>🎞️ Stuck? Peek at the solution</b> (try for 5 minutes first)</summary>

<p><i>Hint: build the 'posts' list with a list comprehension: [{'text': p} for p in posts].</i></p>

```python
SCHEDULE = False   # True = queue in your next free Typefully slot; False = save as a draft only
TYPEFULLY_API = "https://api.typefully.com/v2"

def _typefully(method: str, path: str, **kwargs) -> dict:
    r = httpx.request(method, f"{TYPEFULLY_API}{path}", timeout=30,
                      headers={"Authorization": f"Bearer {TYPEFULLY_API_KEY}"}, **kwargs)
    r.raise_for_status()
    return r.json()

@tool
def create_typefully_draft(
    posts: Annotated[list[str], Field(description="The thread's posts, in order")],
    title: Annotated[str, Field(description="A short internal title for the draft")],
) -> dict:
    """Save an X thread as a Typefully draft (optionally scheduled). Returns the draft's status and URL."""
    payload = {
        "draft_title": title,
        "share": True,
        "platforms": {"x": {"enabled": True, "posts": [{"text": p} for p in posts]}},
    }
    if SCHEDULE:
        payload["publish_at"] = "next-free-slot"

    if not TYPEFULLY_API_KEY:
        return {"status": "dry-run", "would_send": payload}
    social_set_id = get_secret("TYPEFULLY_SOCIAL_SET_ID") or _typefully("GET", "/social-sets")["results"][0]["id"]
    draft = _typefully("POST", f"/social-sets/{social_set_id}/drafts", json=payload)
    return {"status": draft["status"], "url": draft.get("share_url") or draft.get("private_url")}
```

</details>

### ▶️ Cast the Publisher and hold a rehearsal
The Publisher gets a clean brief: a JSON with a `title` and the `posts`. For the rehearsal we hand it the
**final cut from SC 4** and switch Typefully to **dry-run**, so nothing leaves the notebook yet.

In [ ]:
publisher = Agent(
    client,
    name="Publisher",
    instructions=(
        "You are the Publisher. You receive an approved X thread as JSON with a `title` and a `posts` list. "
        "Save it as a draft on Typefully exactly once, with the posts unchanged and the title. "
        "Then reply with one line: the draft's status and its URL (or 'dry-run')."
    ),
    tools=[create_typefully_draft],
)

def show_shipping(response) -> None:
    """Print what the Typefully tool returned inside an agent response."""
    results = [c.result for m in response.messages for c in m.contents if c.type == "function_result"]
    if not results:
        print("⚠️ The Publisher never called the tool. Re-run the cell, or tighten its instructions.")
        return
    out = results[-1]
    if isinstance(out, str):
        try:
            out = json.loads(out)
        except ValueError:
            pass
    print("📦 Typefully tool returned:")
    print(json.dumps(out, indent=2, default=str)[:1500])

_real_key, TYPEFULLY_API_KEY = TYPEFULLY_API_KEY, ""     # 🎭 rehearsal: dry-run only
try:
    rehearsal = await publisher.run(json.dumps({"title": "Rehearsal", "posts": final_thread.posts}))
finally:
    TYPEFULLY_API_KEY = _real_key
show_tool_calls(rehearsal)
show_shipping(rehearsal)

🎟️ **Checkpoint:** you should see one `create_typefully_draft` call and a `dry-run` payload whose
`platforms → x → posts` holds your thread. The real premiere happens in the next scene, with the whole crew.

> 🛡️ **Before you ever set `SCHEDULE = True`:** in production a tool that *posts publicly* should require
> **human approval**. MAF supports that natively with `@tool(approval_mode="always_require")`: the agent pauses and
> asks you before the call runs. Also read [X's automation rules](https://help.x.com/en/rules-and-policies/x-automation).

---
# 🎞️ SC 6 · Premiere: the whole crew in one workflow

So far *you* were the glue code, calling one agent after another. A **workflow** makes that explicit:
a graph of **executors** joined by **edges**. `WorkflowBuilder.add_chain([...])` lines steps up in a
**sequential pipeline**, and `add_edge(a, b)` adds any other connection, including one that points **back**.

```
URL ─▶ 📖 Reader ─▶ ✍️ Writer ─▶ 🧐 Critic ─▶ 🎬 Director ─✅ print it─▶ 📣 Publisher ─▶ Typefully
                        ▲                          │
                        └──────── ❌ retake ───────┘   (3 takes max, then 🛑 nothing ships)
```

An executor doesn't have to be an agent. The **Director** is a plain Python function that makes the call
after every Critic review, with the same rules as your SC 4 loop:
- ✅ **Approved** (and the house-rules check passes) → hand the Publisher a clean brief: a `title` and the `final_posts`.
- ❌ **Notes, takes left** → send the Writer back to set with the notes.
- 🛑 **Notes after take 3** → stop. Nothing gets published; a human should look at it.

In [ ]:
from agent_framework import executor, WorkflowContext, AgentExecutorRequest, AgentExecutorResponse, Message

MAX_TAKES = 3
NOTES_TAG = "🎬 DIRECTOR'S NOTES"

@executor(id="Director")
async def director(critic_turn: AgentExecutorResponse,
                   ctx: WorkflowContext[AgentExecutorRequest, str | Review]) -> None:
    """After each review: ship the approved cut to the Publisher, or send the Writer back for a retake."""
    review = Review.model_validate_json(critic_turn.agent_response.text)
    hard = check_thread_rules(review.final_posts)                    # code has the final say
    approved = review.approved and hard["ok"]
    history = critic_turn.full_conversation                          # brief, every take, every review
    take = 1 + sum(m.role == "user" and (m.text or "").startswith(NOTES_TAG) for m in history)

    verdict = "✅ approved" if approved else "❌ notes"
    lines = [f"   • {n}" for n in hard["issues"]] + [f"   {'💬' if review.approved else '•'} {n}" for n in review.notes]
    await ctx.yield_output("\n".join([f"🎬 Take {take}: {verdict}", *lines]))

    if approved:                                                      # ✅ print it
        await ctx.yield_output(review)
        title = review.final_posts[0][:60].rstrip()
        brief = json.dumps({"title": title, "posts": review.final_posts})
        await ctx.send_message(AgentExecutorRequest(messages=[Message("user", [brief])]), target_id="Publisher")
    elif take < MAX_TAKES:                                            # ❌ retake
        fixes = "\n".join(hard["issues"] + ([] if review.approved else review.notes))
        retake = Message("user", [f"{NOTES_TAG} after take {take}:\n{fixes}\n\nRewrite the thread to fix every note."])
        await ctx.send_message(AgentExecutorRequest(messages=[*history, retake]), target_id="Writer")
    else:                                                             # 🛑 out of takes
        await ctx.yield_output(f"🛑 Out of takes after {MAX_TAKES}. Nothing was published; a human should look at this one.")

print("✅ The Director is ready to call the shots")

### ✍️ YOUR TURN: wire the crew
- `start_executor`: who gets the user's message first?
- `add_chain([...])`: the main line, from the Reader to the Director
- two `add_edge(...)` calls: the Director's two exits, **print it** (to the Publisher) and **retake** (back to the Writer)
- `output_from=[...]`: whose outputs should the workflow return? We want the **Director's calls** and the **Publisher's reply**.

In [ ]:
crew = (
    WorkflowBuilder(
        name="Blog-to-Thread Crew",
        description="Reads a blog URL, writes a fact-checked X thread (up to 3 takes) and saves it as a Typefully draft",
        start_executor=...,      # ✍️ TODO
        output_from=[...],       # ✍️ TODO: the Director and the Publisher
    )
    .add_chain([...])            # ✍️ TODO: reader → writer → critic → director
    .add_edge(..., ...)          # ✍️ TODO: ✅ print it
    .add_edge(..., ...)          # ✍️ TODO: ❌ retake
    .build()
)

<details>
<summary><b>🎞️ Stuck? Peek at the solution</b> (try for 5 minutes first)</summary>

```python
crew = (
    WorkflowBuilder(
        name="Blog-to-Thread Crew",
        description="Reads a blog URL, writes a fact-checked X thread (up to 3 takes) and saves it as a Typefully draft",
        start_executor=reader,
        output_from=[director, publisher],
    )
    .add_chain([reader, writer, critic, director])
    .add_edge(director, publisher)      # ✅ print it
    .add_edge(director, writer)         # ❌ retake: an edge back makes a loop
    .build()
)
```

</details>

In [ ]:
result = await crew.run(f"Turn this blog post into an X thread: {BLOG_URL}")

shipped = None
for event in result:
    if event.type == "executor_invoked":
        print(f"🎬 {event.executor_id} is on set")
    elif event.type == "output":
        if isinstance(event.data, str):
            print(event.data)
        elif isinstance(event.data, Review):
            storyboard(event.data.final_posts, "Crew cut")
        else:
            shipped = event.data

if shipped:
    show_tool_calls(shipped)
    show_shipping(shipped)
    print("\n📣", shipped.text or "(the Publisher had nothing more to say)")

🎟️ **Checkpoint:** you should see the crew on set, one line per take from the Director, the storyboard, and a
Typefully URL (or a dry-run payload). Open the URL to see your thread in Typefully. 🎉 **It's a premiere!**
If the Critic never approved, you'll see 🛑 instead and nothing ships. That's the Director doing its job.

> 🎤 **Pause & ponder:** Which **patterns** do you see? (A sequential pipeline with a reflection loop inside it.)
> In SC 4 the loop was a Python `for`; here it's an **edge back**. What do you gain? (Hint: DevUI is next.)
> Out of takes, we ship nothing. Would you rather ship the best effort, or pause and ask a human?
> MAF also has *concurrent*, *handoff* and *group-chat* orchestrations; which one would add a LinkedIn version in parallel?

---
# 🖥️ SC 7 · Dailies: watch it in DevUI

**DevUI** is MAF's "video village": a web UI where you can chat with any agent or workflow and watch
**every message, tool call and executor** live. One line: `serve(entities=[...])`.

We start it in a background thread so the notebook stays usable, and show it **right here in the cell output**.

In [ ]:
import threading, time

def start_devui(entities, port: int = 8090, height: int = 820):
    from agent_framework_devui import serve
    from agent_framework_devui._server import DevServer
    # DevUI only answers requests addressed to "localhost". Colab's (private, signed-in) proxy uses
    # its own hostname, so we relax that check. The server is still bound to 127.0.0.1 only.
    DevServer._loopback_allowed_hosts = lambda self: None
    threading.Thread(
        target=serve,
        kwargs=dict(entities=entities, port=port, host="127.0.0.1", auto_open=False, auth_enabled=False),
        daemon=True,
    ).start()
    for _ in range(60):                      # wait for the server to come up
        try:
            if httpx.get(f"http://127.0.0.1:{port}/health", timeout=1).status_code == 200:
                break
        except httpx.HTTPError:
            time.sleep(0.5)
    if IN_COLAB:
        from IPython.display import clear_output
        from google.colab import output
        clear_output()
        output.serve_kernel_port_as_iframe(port, height=height)
    else:
        display(HTML(f"<a href='http://127.0.0.1:{port}' target='_blank'>🎬 Open DevUI at http://127.0.0.1:{port}</a>"))

start_devui([crew, reader, writer, critic, publisher])

### 🎬 Things to try in DevUI
The **dropdown at the top left** picks what you're talking to: the whole crew (a workflow) or a single agent.

1. **Chat with one agent.** Pick **Reader**, paste a blog URL into the message box and send it. The panel on the
   right shows the **`scrape_url` tool call**, its result, and the reply as it streams.
2. **Run the whole crew.** Pick **Blog-to-Thread Crew** and click **Configure & Run**. Set **role** to `user`, paste
   `Turn this blog post into an X thread: <your URL>` into **contents**, and hit **Run**.
3. Watch the **graph light up**: Reader → Writer → Critic → Director → Publisher (or back to the Writer for a retake). Click any step to see what it
   received and returned, including the Publisher's **`create_typefully_draft`** call and your draft's URL.
4. Back to single agents: ask the **Writer** to rewrite a hook "for a Gen-Z audience", or paste a thread into the
   **Critic** and see if it approves.

> 🔁 Re-ran a cell that redefines an agent? DevUI still holds the *old* object.
> Call `start_devui([...], port=8091)` to open a fresh one.

---
# ⚖️ SC 8 · The reviews are in: evaluate the Writer

The thread is out. Now the critics weigh in. "It looked good on my blog" is a vibe check, not an evaluation, so we score the
**Writer** on a **test set**: three fixed briefs with known facts, **plus every brief you generated in SC 2**
(saved in `MY_BRIEFS`). Try a few different blog URLs in SC 2 to grow your set. We use three layers of critics:



### 🎯 What we measure, and what "good" looks like

| Check | Critic | Pass bar | If it fails… |
|---|---|---|---|
| 5–7 posts | 🧮 code | every brief | tighten the instructions, or post-process in code |
| ≤ 280 chars per post | 🧮 code | every brief | same: never trust the model to count |
| ≤ 2 hashtags | 🧮 code | every brief | same |
| **Faithfulness** ≥ 4/5 | ⚖️ LLM judge | every brief | 🚨 a hallucination. Top priority: fix before anything else. |
| **Hook strength** ≥ 4/5 | ⚖️ LLM judge | most briefs | subjective: read the judge's reasons, then iterate on the brief |

We use MAF's built-in eval helpers: wrap each check with **`@evaluator`**, bundle them in a **`LocalEvaluator`**,
and **`evaluate_agent(...)`** runs the agent on every test query and scores each response.

In [ ]:
from agent_framework import evaluate_agent, evaluator, LocalEvaluator

# A small, fixed test set: three briefs with known facts (in real life: 20-50, from real traffic)
EVAL_BRIEFS = [
    """TITLE: What my first internship actually taught me
URL: https://myblog.dev/first-internship
THESIS: Writing things down mattered more than clever code.
KEY POINTS:
- 12-week software internship at a 40-person fintech startup in Bengaluru
- Shipped 3 features; the biggest cut invoice processing from 2 days to 4 hours
- Manager's advice: ask after 30 minutes stuck, not after 3 hours
- Broke the staging database once; wrote a 1-page post-mortem and a backup script
- First pull request got 47 review comments; by week 10 was reviewing others' code
- Got a return offer""",
    """TITLE: I built a chatbot over my college notes in one weekend
URL: https://myblog.dev/notes-chatbot
THESIS: Retrieval beats a bigger model when the answers live in your own notes.
KEY POINTS:
- 220 pages of handwritten notes, scanned and OCR'd with a free tool
- Split into 1,100 chunks and stored in a local vector database
- Answered 41 of 50 past exam questions correctly, vs 23 without retrieval
- Total cost: zero rupees, everything ran on a laptop
- Biggest bug: OCR read 'O(n log n)' as 'O(n log m)'""",
    """TITLE: 30 days without Instagram
URL: https://myblog.dev/no-instagram
THESIS: Boredom came back, and it was the best part.
KEY POINTS:
- Screen time dropped from 5h 10m to 2h 05m a day
- Read 4 books in the month, versus 1 in the previous six months
- Days 3-5 were the hardest; reached for the phone about 60 times a day
- Started journaling for 10 minutes every night
- Reinstalled after 30 days but turned off all notifications""",
]

def _parse(response: str) -> Thread | None:
    try:
        return Thread.model_validate_json(response)
    except Exception:
        return None

@evaluator(name="📏 5-7 posts")
def post_count(response: str) -> dict:
    t = _parse(response)
    n = len(t.posts) if t else 0
    return {"passed": 5 <= n <= 7, "reason": f"{n} posts"}

@evaluator(name="✂️ ≤280 chars")
def post_length(response: str) -> dict:
    t = _parse(response)
    longest = max((len(p) for p in t.posts), default=0) if t else 999
    return {"passed": longest <= 280, "reason": f"longest post: {longest} chars"}

@evaluator(name="#️⃣ ≤2 hashtags")
def hashtag_budget(response: str) -> dict:
    t = _parse(response)
    n = len(re.findall(r"#\w+", " ".join(t.posts))) if t else 0
    return {"passed": n <= 2, "reason": f"{n} hashtags"}

### ✍️ YOUR TURN: write the judge's rubric
An LLM judge is only as good as its **rubric**. Vague rubrics ("rate the quality 1-10") give noisy, inflated scores.
Good rubrics define **each score level** with concrete anchors. Fill in the anchors for both criteria.

In [ ]:
class JudgeScore(BaseModel):
    faithfulness: int = Field(ge=1, le=5, description="1-5, see rubric")
    hook: int = Field(ge=1, le=5, description="1-5, see rubric")
    reason: str = Field(description="One sentence citing the weakest post")

JUDGE_INSTRUCTIONS = """
You are a strict evaluator of X threads written from a reader's brief.
Score two criteria from 1 to 5. Be harsh; a 5 should be rare.

FAITHFULNESS: are all claims supported by the brief?
5 = every fact, number and quote appears in the brief
4 = TODO
3 = TODO
2 = TODO
1 = several invented or contradicting facts

HOOK: would post 1 make a scrolling reader stop and pay attention?
A strong hook is memorable: provocative, a sharp question, an exclamation or a quotable line.
5 = TODO
4 = TODO
3 = TODO
2 = generic ("Here's what I learned...")
1 = TODO

In `reason`, cite the post number that pulled the score down.
"""

judge = Agent(client, name="Judge", instructions=JUDGE_INSTRUCTIONS,
              default_options={"response_format": JudgeScore})

<details>
<summary><b>🎞️ Stuck? Peek at the solution</b> (try for 5 minutes first)</summary>

<p><i>Hint: anchor each level on something observable: 'an invented number', 'a generic opener', 'starts with Thread:'.</i></p>

```python
class JudgeScore(BaseModel):
    faithfulness: int = Field(ge=1, le=5, description="1-5, see rubric")
    hook: int = Field(ge=1, le=5, description="1-5, see rubric")
    reason: str = Field(description="One sentence citing the weakest post")

JUDGE_INSTRUCTIONS = """
You are a strict evaluator of X threads written from a reader's brief.
Score two criteria from 1 to 5. Be harsh; a 5 should be rare.

FAITHFULNESS: are all claims supported by the brief?
5 = every fact, number and quote appears in the brief
4 = one harmless paraphrase or generalisation, no new facts
3 = one claim not in the brief, but plausible
2 = an invented number, name or quote
1 = several invented or contradicting facts

HOOK: would post 1 make a scrolling reader stop and pay attention?
A strong hook is memorable: provocative, a sharp question, an exclamation or a quotable line.
5 = memorable and specific: a provocative claim, question or quotable line that demands the next post
4 = specific and interesting, but not memorable enough to quote or repeat
3 = a clear topic statement with no tension or curiosity
2 = generic ("Here's what I learned...")
1 = no hook at all, or starts with "Thread:" / 🧵

In `reason`, cite the post number that pulled the score down.
"""

judge = Agent(client, name="Judge", instructions=JUDGE_INSTRUCTIONS,
              default_options={"response_format": JudgeScore})
```

</details>

In [ ]:
_verdicts: dict = {}

async def judge_once(query: str, response: str) -> JudgeScore:
    """One judge call per (brief, thread), shared by both judge checks below."""
    key = (query, response)
    if key not in _verdicts:
        _verdicts[key] = asyncio.ensure_future(judge.run(f"BRIEF:\n{query}\n\nTHREAD (JSON):\n{response}"))
    return (await _verdicts[key]).value

@evaluator(name="⚖️ judge: faithful ≥4")
async def faithfulness(query: str, response: str) -> dict:
    s = await judge_once(query, response)
    why = f" · {s.reason}" if s.faithfulness <= s.hook else ""
    return {"passed": s.faithfulness >= 4, "reason": f"{s.faithfulness}/5{why}"}

@evaluator(name="⚖️ judge: hook ≥4")
async def hook_strength(query: str, response: str) -> dict:
    s = await judge_once(query, response)
    why = f" · {s.reason}" if s.hook < s.faithfulness else ""
    return {"passed": s.hook >= 4, "reason": f"{s.hook}/5{why}"}

def _title(brief_text: str) -> str:
    m = re.search(r"TITLE:\s*(.+)", brief_text)
    return (m.group(1) if m else brief_text.strip().splitlines()[0]).strip(" *")

def show_scorecard(results) -> None:
    from html import escape
    cell = "padding:8px 10px;border:1px solid #E4DCE6;vertical-align:top;white-space:normal;overflow-wrap:anywhere"
    rows = ""
    for item in results.items:
        tag = " <span style='color:#B0477A;font-size:11px'>(yours)</span>" if item.input_text in globals().get("MY_BRIEFS", []) else ""
        cells = "".join(
            f"<td style='{cell};min-width:{240 if 'judge' in s.name else 120}px'>"
            f"{'✅' if s.passed else '❌'} <span style='color:#5B4A60;font-size:12px'>"
            f"{escape(str((s.sample or {}).get('reason', '')))}</span></td>"
            for s in item.scores
        )
        rows += f"<tr><td style='{cell};min-width:200px'><b>{escape(_title(item.input_text))}</b>{tag}</td>{cells}</tr>"
    head = "".join(f"<th style='{cell};text-align:left'>{escape(s.name)}</th>" for s in results.items[0].scores)
    rc = results.result_counts
    display(HTML(
        f"<table style='border-collapse:collapse;font:13px sans-serif;table-layout:auto;width:100%'>"
        f"<tr><th style='{cell};text-align:left'>Brief</th>{head}</tr>{rows}</table>"
        f"<p><b>{rc['passed']}/{results.total} briefs passed every check.</b></p>"
    ))

TEST_SET = EVAL_BRIEFS + [b for b in globals().get("MY_BRIEFS", []) if b not in EVAL_BRIEFS]
print(f"🎞️ Scoring the Writer on {len(TEST_SET)} briefs ({len(TEST_SET) - len(EVAL_BRIEFS)} of them yours)…")

[results] = await evaluate_agent(
    agent=writer,
    queries=TEST_SET,
    evaluators=LocalEvaluator(post_count, post_length, hashtag_budget, faithfulness, hook_strength),
)
show_scorecard(results)

### 🔍 Reading your scorecard
- **Start with the ❌s and read the reasons.** A score without a reason is just a number; the reason tells you what to fix.
- **Faithfulness first.** A thread with a great hook and an invented number is worse than a boring, true one.
- **Code checks failing?** Don't only shout louder in the prompt. Add a guard in code (you already built one in SC 4!).
- **Now iterate:** change *one* line in `WRITER_INSTRUCTIONS`, re-run SC 3's writer cell and this cell, and compare.
  That's **eval-driven development**: every prompt change is a measurable experiment.

> 🎤 **Pause & ponder:** our judge is the **same model** as the writer, it sees only a handful of briefs, and we never checked it
> against a human. Name one way each of these could fool you.
> (Hint: self-preference bias, tiny sample, uncalibrated judge.)

<details>
<summary><b>🎬 Director's cut (optional): Azure AI Evaluation SDK</b></summary>

For production-grade evals, the [Azure AI Evaluation SDK](https://learn.microsoft.com/azure/ai-foundry/how-to/develop/evaluate-sdk)
ships calibrated, research-backed evaluators (`GroundednessEvaluator`, `RelevanceEvaluator`, `CoherenceEvaluator`, plus safety
evaluators) and logs runs to Azure AI Foundry. It works with any OpenAI-compatible endpoint:

```python
%pip install -q azure-ai-evaluation
from azure.ai.evaluation import GroundednessEvaluator, OpenAIModelConfiguration

model_config = OpenAIModelConfiguration(type="openai", model=MODEL, base_url=BASE_URL, api_key=API_KEY)
groundedness = GroundednessEvaluator(model_config)
print(groundedness(response=" ".join(final_thread.posts), context=brief))
```
</details>

🎬 **That's a wrap!** Your crew reads, writes, critiques, ships, and gets reviewed.

---
# 🎞️ Credits roll

### What you built tonight
| Scene | Concept | MAF building block |
|---|---|---|
| SC 1 | A model behind an OpenAI-compatible endpoint | `OpenAIChatCompletionClient` |
| SC 2 | Agent = model + instructions + tools, running a loop | `Agent`, `@tool` |
| SC 3 | Structured output | `response_format=PydanticModel` |
| SC 4 | Code checks as tools + the reflection pattern | `tool(fn)`, a bounded Python loop |
| SC 5 | Real-world side effects, safely | `@tool` on the Typefully API, drafts and dry-runs first |
| SC 6 | Multi-agent workflow with a retake loop and a code step | `WorkflowBuilder().add_chain()`, `add_edge()`, `@executor` |
| SC 7 | Observability for developers | DevUI `serve()` |
| SC 8 | Evals: code checks + LLM-as-judge | `@evaluator`, `LocalEvaluator`, `evaluate_agent` |

### 🎬 Director's cut: ideas to keep going
- 🔀 **Concurrent:** add a *LinkedIn Writer* that runs in parallel with the X Writer (`add_fan_out_edges`).
- 🙋 **Human-in-the-loop:** mark `create_typefully_draft` with `approval_mode="always_require"`.
- 🔌 **MCP:** swap `scrape_url` for a Firecrawl **MCP server** via `MCPStreamableHTTPTool`.
- 📈 **Grow the test set** to 20 real blogs, and track the scorecard after every prompt change.
- 🔭 **Observability:** turn on OpenTelemetry tracing and watch token cost per agent.

### 📚 References
- Microsoft Agent Framework: [docs](https://learn.microsoft.com/agent-framework/) · [GitHub](https://github.com/microsoft/agent-framework)
- [Gemini API (OpenAI compatibility)](https://ai.google.dev/gemini-api/docs/openai) · [Firecrawl](https://docs.firecrawl.dev) · [Typefully API](https://typefully.com/docs/api)
- [Building effective agents](https://www.anthropic.com/engineering/building-effective-agents) (Anthropic)

## 🍿 Stay for the post-credits scene: the quiz is up on screen!